In [1]:
import pandas as pd 
import os 
import logging 
import traceback
from basicprocess import create_folder, outputlog, findfiles, read_combined_dataframe
from TDXdataframe import read_businfo_xml

In [20]:
def get_taipeibusreport(odspath):

    logging.info("開始讀取台北市營運月報")
    logging.info(f"台北市公車營運月報的原始檔案為{odspath}")

    sheet_names = pd.ExcelFile(odspath).sheet_names
    dfs = []
    for sheet in sheet_names:
        odsdf = pd.read_excel(odspath, sheet_name=sheet, engine='odf')
        odsdf['Time'] = sheet
        dfs.append(odsdf)

    df = pd.concat(dfs, ignore_index=True)

    logging.info("台北市營運月報整併完成")
    

    df['年'] = df['Time'].str[:3].astype(int) + 1911
    df['月'] = df['Time'].str[3:].astype(int)
    # df['資料時間'] = pd.to_datetime(df['年'].astype(str) + '-' + df['月'].astype(str)).dt.to_period('M')
    df['資料時間'] = pd.to_datetime(
        df['年'].astype(str) + '-' + df['月'].astype(str).str.zfill(2),
        format='%Y-%m',
        errors='raise').dt.to_period('M')

    cols = ['資料時間'] + [c for c in df.columns if c not in ['年', '月', 'Time', '資料時間']]
    df = df.reindex(columns=cols)

    logging.info("輸出指定格式")

    rename_dist = {'資料時間':'Month',
                '客運業者':'OperatorName', 
                '路線代碼':'RouteID', 
                '路線別':'RouteName', 
                '總班次':'Shifts', 
                '總載客人次':'Passenger',
                '總行駛里程':'Miles',
                '延人公里':'PaxKm', 
                '總營收': 'Revenue'}
    df = df[list(rename_dist)]
    df = df.rename(columns = rename_dist)

    return df

def check_if_morethanone(df, checklists, timecolumn, warningfolder, dataname = '資料'):
    """
    檢查每個 timecolumn 內，checklists 是否有重複值

    Parameters
    ----------
    df : pandas.DataFrame
    checklists : list
        需要檢查是否重複的欄位
    timecolumn : str
        時間欄位（例如 Month）

    Returns
    -------
    duplicated_df : pandas.DataFrame
        含有重複資料的 dataframe（只保留重複者）
    summary : pandas.DataFrame
        每個月份重複筆數的摘要
    """

    # 找出在「同一個月 + checklists」下重複的資料
    mask = df.duplicated(subset=[timecolumn] + checklists, keep=False)
    duplicated_df = df[mask].sort_values([timecolumn] + checklists)

    if len(duplicated_df) > 0:
        logging.warning(f"{dataname} 有重複的資料")

        # 每月重複筆數摘要
        summary = (
            duplicated_df
            .groupby(timecolumn)
            .size()
            .reset_index(name='DuplicatedRows')
        )

        outputfile = os.path.join(warningfolder, f"{dataname}重複資料.xlsx")

        with pd.ExcelWriter(outputfile, engine='xlsxwriter') as writer:
            duplicated_df.to_excel(writer, index=True, sheet_name='有重複的資料')
            summary.to_excel(writer, index=True, sheet_name='每月重複筆數')

        logging.info(f"{dataname}路線營運月報，路徑：{outputfile}")

        return False     

    else:
        logging.info(f"{dataname}在{checklists}的組合底下沒有重複資料")
        return True

def get_businfo():
    businfos = []
    for xmlpath in findfiles(filefolderpath=os.path.join(os.getcwd(), '..', '00_TDX資料下載', '03公車路線營運資料'), filetype='xml'):
        businfo = read_businfo_xml(xml_path=xmlpath)
        businfos.append(businfo)
    businfo = pd.concat(businfos)

    return businfo

In [3]:
'''Setup & Main Execution'''
# 00_Setup 所有全域函數
logfile = os.path.abspath(os.path.join(os.getcwd(), '..', 'Log', '04_營運月報整理.log'))
if os.path.exists(logfile):
    os.remove(logfile)
    
logging.basicConfig(
    filename=logfile,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

def main():
    logging.info('Start Processing...')

    outputfolder = create_folder(os.path.join(os.getcwd(), '..', '03_處理後資料'))
    monthlyreport_organized_folder = create_folder(os.path.join(outputfolder, '03_公車營運月報'))
    finalorganizedfile = os.path.join(monthlyreport_organized_folder, '月報統計數量.xlsx')
    
    # 處理台北公車路線資料
    taipeidf = get_taipeibusreport(r"D:\OneDrive - 鼎漢國際工程顧問股份有限公司\部門空間管理者8\B-6812-修訂臺北都會區整體路網\Technical\18票證月報資料\04臺北\01公運處\附2-項目(二)臺北市營運資料(市區公車)v1.ods")
    taipeidf.to_excel(os.path.join(monthlyreport_organized_folder, '臺北市公車營運月報.xlsx'), index = False)
    logging.info('輸出臺北市公車營運月報整理結果')
    temp = check_if_morethanone(df = taipeidf, 
                                checklists = ['RouteID', 'RouteName' ], 
                                timecolumn = 'Month', 
                                dataname = '臺北市公車營運月報', 
                                warningfolder=create_folder(os.path.join(monthlyreport_organized_folder, 'error')))
    if temp == False:
        logging.info("臺北市公車因為有多個OpeartionName經營同一條路線")
        taipeidf = taipeidf.drop(columns='OperatorName').groupby(['Month','RouteName']).agg({'Shifts':'sum',
                                                                                             'Passenger':'sum',
                                                                                             'Miles':'sum',
                                                                                             'PaxKm':'sum',
                                                                                             'Revenue':'sum'}).reset_index()
        taipeidf.to_excel(os.path.join(monthlyreport_organized_folder, '臺北市公車營運月報.xlsx'), index = False)
        logging.info('重新輸出臺北市公車營運月報整理結果')

    del temp

    alldf = read_combined_dataframe(findfiles(monthlyreport_organized_folder, 'xlsx', recursive=False))
    alldf.to_excel(finalorganizedfile)


    logging.info('Finished Processing.')


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        logging.error("main() 執行失敗：%s", e)  
        logging.error("Traceback:\n%s", traceback.format_exc())
    
    outputlog(logfile=logfile)

In [31]:
outputfolder = create_folder(os.path.join(os.getcwd(), '..', '03_處理後資料'))
monthlyreport_organized_folder = create_folder(os.path.join(outputfolder, '03_公車營運月報'))
taipeidf = read_combined_dataframe(findfiles(monthlyreport_organized_folder, 'xlsx', recursive=False), filepath=False)

In [32]:
businfo = get_businfo()

In [33]:
taipeidf = taipeidf.merge(businfo.reindex(columns = ['RouteUID', 'RouteNameZh','SubRouteUID', 'SubRouteNameZh']).drop_duplicates(), left_on=['RouteName'], right_on=['SubRouteNameZh'], how = 'left')

In [39]:
taipeidf[~taipeidf['SubRouteNameZh'].isna()].drop_duplicates(subset = ['RouteName'])

,Month,RouteName,Shifts,Passenger,Miles,PaxKm,Revenue,RouteUID,RouteNameZh,SubRouteUID,SubRouteNameZh
132,2024-01,212夜,186,3395,3199,23358,80053,TPE16132,212夜,TPE157442,212夜
134,2024-01,212直,4900,148215,84271,1019719,3494473,TPE10911,212直,TPE10911,212直
137,2024-01,218區,1894,22856,8427,77710,538881,TPE11158,218區,TPE111580,218區
138,2024-01,218直,44,1537,777,10851,36237,TPE11157,218直,TPE111570,218直
140,2024-01,225區,62,2425,728,11398,57181,TPE16128,225區,TPE155898,225區
...,...,...,...,...,...,...,...,...,...,...,...
3940,2024-10,藍15,5012,197352,66772,2107719,4649729,NWT16296,藍15,NWT157291,藍15
3943,2024-10,藍26,2462,91192,39023,1156315,2148538,TPE10432,藍26,TPE10432,藍26
3948,2024-10,藍5,5166,101773,33049,521078,2397831,TPE10831,藍5,TPE10831,藍5
3951,2024-10,藍7,1909,58159,22526,549021,1367298,TPE15361,藍7,TPE157468,藍7


In [38]:
businfo[businfo['SubRouteNameZh'].str.contains('536')].reindex(columns = ['RouteUID', 'RouteNameZh','SubRouteUID', 'SubRouteNameZh'])

,RouteUID,RouteNameZh,SubRouteUID,SubRouteNameZh
28,TPE10263,536,TPE102630,536
29,TPE10263,536,TPE102630,536
1147,TPE19720,536區,TPE162405,536區
1148,TPE19720,536區,TPE162405,536區
